# Module 1: 16S Pipeline with DADA2 and QIIME2

In this notebook we will process 16S rRNA reads from raw FASTQ files to obtain:
- An ASV table
- Taxonomic assignment
- Alpha and beta diversity metrics
- Taxonomic plots

## Pipeline overview

```
Raw FASTQ
    └─► Import into QIIME2
            └─► Visualize quality
                    └─► Remove primers (cutadapt)
                            └─► Denoising (DADA2)
                                    └─► Phylogeny
                                            └─► Alpha and beta diversity
                                                    └─► Taxonomy
```

## Requirements

Make sure the environment is active before opening this notebook:

```bash
conda activate qiime2-amplicon-2025.4
jupyter notebook
```

## 0. Path configuration

We define all paths in one place to make it easy to adapt this notebook to a different dataset.

In [ ]:
import os

# --- Paths ---
DATA_DIR      = "../data/raw_reads"
RESULTS_DIR   = "../results"
METADATA      = "../data/metadata.tsv"
MANIFEST      = "../data/manifest.tsv"
CLASSIFIER    = "../data/taxonomy_db/silva-138-99-nb-classifier.qza"

# --- 16S Primers (V4 region, 515F/806R) ---
FWD_PRIMER  = "GTGYCAGCMGCCGCGGTAA"    # 515F (forward)
REV_PRIMER  = "GGACTACHVGGGTWTCTAAT"   # 806R (reverse)
FWD_ADAPTER = "ATTAGAWACCCBNGTAGTCC"   # RC of reverse primer (adapter on R1)
REV_ADAPTER = "TTACCGCGGCKGCTGRCAC"    # RC of forward primer (adapter on R2)

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Paths configured")

## 1. Manifest and metadata

QIIME2 needs a **manifest** to locate the FASTQ files for each sample, and a **metadata** file with information about the samples.

### Manifest
Three columns: `sample-id`, `forward-absolute-filepath`, `reverse-absolute-filepath`.

### Metadata
One row per sample with variables of interest (treatment, timepoint, etc.).

We will generate the manifest automatically from the files in `raw_reads/`.

In [ ]:
import pandas as pd
import glob

# --- Generate manifest ---
raw_reads_abs = os.path.abspath(DATA_DIR)
r1_files = sorted(glob.glob(os.path.join(raw_reads_abs, "*_R1.fastq.gz")))

rows = []
for r1 in r1_files:
    sample_id = os.path.basename(r1).replace("_R1.fastq.gz", "")
    r2 = r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    rows.append({"sample-id": sample_id,
                 "forward-absolute-filepath": r1,
                 "reverse-absolute-filepath": r2})

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST, sep="\t", index=False)
print("Manifest generated:")
manifest

In [ ]:
metadata = pd.read_csv(METADATA, sep="\t")
print("Metadata:")
metadata

## 2. Import sequences into QIIME2

QIIME2 works with `.qza` (QIIME2 Artifact) files. The first step is to import the FASTQs using the manifest we generated.

- **Type:** `SampleData[PairedEndSequencesWithQuality]`
- **Format:** `PairedEndFastqManifestPhred33V2`

In [ ]:
!qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path {MANIFEST} \
    --output-path {RESULTS_DIR}/sequences.qza \
    --input-format PairedEndFastqManifestPhred33V2

print("✓ Sequences imported:", RESULTS_DIR + "/sequences.qza")

## 3. Visualize read quality

Before denoising we need to inspect read quality to decide truncation parameters.

The `.qzv` file can be visualized at [https://view.qiime2.org](https://view.qiime2.org).

In [ ]:
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences.qza \
    --o-visualization {RESULTS_DIR}/sequences_summary.qzv

print("✓ Quality summary generated")
print("  → Visualize at https://view.qiime2.org — upload:", RESULTS_DIR + "/sequences_summary.qzv")

### What to look for in the quality plot

- X axis: position in the read (bp)
- Y axis: Phred quality score (Q)
- **Q ≥ 30** = 99.9% accuracy (good quality)
- **Q < 20** = low quality zone, consider truncating

Use this plot to set `--p-trunc-len-f` and `--p-trunc-len-r` in the denoising step.

## 4. Remove primers with Cutadapt

Primers must be removed before denoising. If they are not removed, DADA2 may interpret them as real biological variation and generate spurious ASVs.

We also remove adapter sequences (reverse complements of the primers) and run cutadapt twice (`--p-times 2`) to catch any remaining primer dimers.

In [ ]:
!qiime cutadapt trim-paired \
    --i-demultiplexed-sequences {RESULTS_DIR}/sequences.qza \
    --p-front-f {FWD_PRIMER} \
    --p-front-r {REV_PRIMER} \
    --p-adapter-f {FWD_ADAPTER} \
    --p-adapter-r {REV_ADAPTER} \
    --p-times 2 \
    --p-discard-untrimmed \
    --p-cores 4 \
    --o-trimmed-sequences {RESULTS_DIR}/sequences_trimmed.qza \
    --verbose 2>&1 | tail -20

print("✓ Primers removed")

In [ ]:
# Visualize quality after trimming
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences_trimmed.qza \
    --o-visualization {RESULTS_DIR}/sequences_trimmed_summary.qzv

print("✓ Visualize post-trimming quality at https://view.qiime2.org")

## 5. Denoising with DADA2

DADA2 does four things in a single step:
1. **Filters** low-quality reads
2. **Learns** the error model of the run
3. **Denoises** and corrects sequencing errors
4. **Removes** chimeras

The result is a set of **ASVs** (Amplicon Sequence Variants) — exact sequences, more precise than OTUs.

### ASVs vs OTUs
| | OTUs | ASVs |
|---|---|---|
| Similarity | 97% | 100% (exact sequence) |
| Resolution | Genus/species | Sub-species |
| Reproducibility | Depends on threshold | High |
| Method | Clustering | Denoising |

### Key parameters
- `--p-trunc-len-f` / `--p-trunc-len-r`: position to truncate reads. Use `0` to skip truncation (DADA2 handles quality internally).
- If quality drops sharply before the end of the read, truncate there — but make sure R1 and R2 still overlap by at least 20bp.

In [ ]:
# trunc-len 0 = no truncation (DADA2 filters by quality internally)
# Adjust if quality drops before the end of the read (see step 3)
TRUNC_F = 0
TRUNC_R = 0

!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs {RESULTS_DIR}/sequences_trimmed.qza \
    --p-trim-left-f 0 \
    --p-trim-left-r 0 \
    --p-trunc-len-f {TRUNC_F} \
    --p-trunc-len-r {TRUNC_R} \
    --p-n-threads 4 \
    --o-table {RESULTS_DIR}/asv_table.qza \
    --o-representative-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-denoising-stats {RESULTS_DIR}/denoising_stats.qza

print("✓ Denoising complete")

## 6. Denoising statistics

Check how many reads passed each filtering step. If many reads are lost at a particular step, it may indicate a problem with the parameters.

In [ ]:
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/denoising_stats.qza \
    --o-visualization {RESULTS_DIR}/denoising_stats.qzv

print("✓ Denoising statistics generated")
print("  → Visualize at https://view.qiime2.org")

### What to expect

| Column | Description | Expected value |
|---|---|---|
| `input` | Total reads | 100% |
| `filtered` | Post quality filter | > 80% |
| `denoised` | Post error correction | ~ filtered |
| `merged` | R1+R2 merged | > 70% |
| `non-chimeric` | Post chimera removal | > 70% of input |

## 7. ASV table and representative sequences

Visualize the ASV table to see how many ASVs and reads we have per sample.

In [ ]:
!qiime feature-table summarize \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --m-sample-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/asv_table_summary.qzv

!qiime feature-table tabulate-seqs \
    --i-data {RESULTS_DIR}/rep_seqs.qza \
    --o-visualization {RESULTS_DIR}/rep_seqs.qzv

print("✓ ASV table and representative sequences generated")

## 8. Phylogeny

We build a phylogenetic tree from the representative sequences. This is required for diversity metrics that account for evolutionary distances (UniFrac).

The `align-to-tree-mafft-fasttree` command does in one step:
1. Multiple sequence alignment with MAFFT
2. Mask uninformative positions
3. Build tree with FastTree
4. Root the tree

In [ ]:
!qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-alignment {RESULTS_DIR}/aligned_rep_seqs.qza \
    --o-masked-alignment {RESULTS_DIR}/masked_aligned_rep_seqs.qza \
    --o-tree {RESULTS_DIR}/unrooted_tree.qza \
    --o-rooted-tree {RESULTS_DIR}/rooted_tree.qza

print("✓ Phylogenetic tree built")

## 9. Alpha and beta diversity

### Alpha diversity
Measures diversity **within** a single sample.
- **Shannon:** accounts for richness and relative abundance
- **Observed features:** number of unique ASVs
- **Faith's PD:** phylogenetic diversity

### Beta diversity
Measures diversity **between** samples.
- **Bray-Curtis:** abundance-based, no phylogeny
- **Unweighted UniFrac:** presence/absence + phylogeny
- **Weighted UniFrac:** abundance + phylogeny

### Rarefaction
Before computing diversity, we normalize by sequencing depth (rarefaction). The `--p-sampling-depth` parameter sets the minimum depth — samples with fewer reads will be excluded.

Check the ASV table summary to choose a value that retains most samples.

In [ ]:
# ⚠️ Adjust based on the minimum reads per sample seen in step 7
SAMPLING_DEPTH = 5000

!qiime diversity core-metrics-phylogenetic \
    --i-phylogeny {RESULTS_DIR}/rooted_tree.qza \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --p-sampling-depth {SAMPLING_DEPTH} \
    --m-metadata-file {METADATA} \
    --output-dir {RESULTS_DIR}/diversity

print("✓ Diversity metrics computed in:", RESULTS_DIR + "/diversity/")

In [ ]:
# Rarefaction curves — check at what depth diversity stabilizes
!qiime diversity alpha-rarefaction \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --i-phylogeny {RESULTS_DIR}/rooted_tree.qza \
    --p-max-depth {SAMPLING_DEPTH} \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/alpha_rarefaction.qzv

print("✓ Rarefaction curves generated")

In [ ]:
# Statistical test: differences in alpha diversity between groups
!qiime diversity alpha-group-significance \
    --i-alpha-diversity {RESULTS_DIR}/diversity/shannon_vector.qza \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/diversity/shannon_significance.qzv

print("✓ Alpha diversity significance test (Shannon) generated")

In [ ]:
# Statistical test: differences in community composition between groups (PERMANOVA)
!qiime diversity beta-group-significance \
    --i-distance-matrix {RESULTS_DIR}/diversity/bray_curtis_distance_matrix.qza \
    --m-metadata-file {METADATA} \
    --m-metadata-column Treatment \
    --o-visualization {RESULTS_DIR}/diversity/bray_curtis_significance.qzv

print("✓ PERMANOVA test (Bray-Curtis) generated")

### PCoA — Sample ordination

PCoA (Principal Coordinates Analysis) displays sample similarity in a reduced space. Samples that cluster together have similar microbial composition.

The `*_emperor.qzv` files in the diversity folder contain the interactive PCoA plots.

In [ ]:
import os
pcoa_files = [f for f in os.listdir(RESULTS_DIR + "/diversity") if "emperor" in f]
print("Available PCoA files:")
for f in pcoa_files:
    print(" →", f)
print("\nVisualize at https://view.qiime2.org")

## 10. Taxonomic assignment

We assign taxonomy to each ASV using a Naive Bayes classifier trained on the Silva 138 database.

The classifier file is ~1.7GB and must be downloaded once before running this step. Run the cell below — it will skip the download if the file already exists.

In [ ]:
import os

CLASSIFIER = "../data/taxonomy_db/silva-138-99-nb-classifier.qza"

if not os.path.exists(CLASSIFIER):
    print("Downloading Silva classifier (~1.7GB) — this may take a few minutes...")
    !wget -q --show-progress \
        -O {CLASSIFIER} \
        https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza
    print("✓ Silva classifier downloaded")
else:
    print("✓ Silva classifier already exists, skipping download")

In [ ]:
CLASSIFIER = "../data/taxonomy_db/silva-138-99-nb-classifier.qza"

!qiime feature-classifier classify-sklearn \
    --i-classifier {CLASSIFIER} \
    --i-reads {RESULTS_DIR}/rep_seqs.qza \
    --p-n-jobs -1 \
    --o-classification {RESULTS_DIR}/taxonomy.qza

print("✓ Taxonomy assigned")

In [ ]:
# Visualize taxonomy table
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/taxonomy.qza \
    --o-visualization {RESULTS_DIR}/taxonomy.qzv

print("✓ Taxonomy table generated")

## 11. Filter contaminants and plot relative abundance (barplots)

Before visualizing taxonomic composition, we remove ASVs assigned to **mitochondria** and **chloroplasts** — these are eukaryotic 16S sequences that do not represent bacterial microbiota.

Barplots show the relative abundance of each sample. You can change the taxonomic level (Phylum, Class, Order, Family, Genus) in the interactive viewer.

In [ ]:
# Filter mitochondria and chloroplasts
!qiime taxa filter-table \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --i-taxonomy {RESULTS_DIR}/taxonomy.qza \
    --p-exclude mitochondria,chloroplast \
    --o-filtered-table {RESULTS_DIR}/asv_table_filtered.qza

print("✓ Mitochondria and chloroplasts removed")

# Barplot with filtered table
!qiime taxa barplot \
    --i-table {RESULTS_DIR}/asv_table_filtered.qza \
    --i-taxonomy {RESULTS_DIR}/taxonomy.qza \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/taxa_barplot.qzv

print("✓ Taxonomic barplot generated")
print("  → Visualize at https://view.qiime2.org")

## 12. Summary of generated files

At the end of the pipeline you should have the following files in `results/`:

In [ ]:
import os

print(f"Contents of {RESULTS_DIR}:\n")
for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f"{indent}{folder}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

## Quick reference: `.qza` vs `.qzv` files

| Extension | Type | Use |
|---|---|---|
| `.qza` | Artifact | Processed data, input for next steps |
| `.qzv` | Visualization | View only at view.qiime2.org |

Both are ZIP files — you can rename them to `.zip` and open them to inspect their contents.